# Supplementary Figure 3 | Concordance of GTEx v10 and v11 expression by biotype

**a.** Per gene × tissue median TPM (across samples in the tissue) in v10 versus v11, stratified and colored by v11 biotype (protein-coding, lncRNA, pseudogene). Each point is the median expression for one gene annotated in both GENCODE releases in a given tissue. Axes are log10(TPM+1). Shaded regions mark strong expression discordance (≥10-fold between releases after a 0.1 TPM pseudocount: |log₁₀((TPMv11+0.1)/(TPMv10+0.1))| ≥ 1); dashed curves show the corresponding fold-change boundaries (curved on these axes) and a solid line marks y=x. Panel titles report Lin’s concordance correlation coefficient (CCC) on log10(TPM+1) and the number of unique genes with ≥1 strongly discordant tissue. Concordance is high overall for protein-coding genes (CCC = 0.996), but 54 are strongly discordant. Many lncRNAs (CCC = 0.970; 212 strongly discordant) and especially pseudogenes (CCC = 0.864; 441 strongly discordant) are strongly discordant.

**b.** For each strongly discordant gene from (a), the change in percent ambiguous exon base pairs (v47 − v39, % of total exon bp in the version that are ambiguous) versus the exon bp unique to one release (plotted as % of v47 exon bp). Strongly discordant genes of all biotypes generally show either annotation change, ambiguity change, or both.


## Setup


In [ ]:
from pathlib import Path
import gzip
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

fig_width, fig_height, fig_dpi = 7.2, 5.0, 300
fold_thresh = 10.0  # strongly discordant if |log10 FC| >= log10(fold_thresh)
tpm_eps = 0.1       # pseudocount for fold change (and near-zero stability)
max_scatter = 200_000  # display downsample; CCC / counts use full data
# seed for display downsampling
np.random.seed(0)

biotype_order = ["protein_coding", "lncRNA", "pseudogene"]
protein_coding_types = {
    "protein_coding", "IG_C_gene", "IG_D_gene", "IG_J_gene", "IG_V_gene",
    "TR_C_gene", "TR_D_gene", "TR_J_gene", "TR_V_gene",
}
pseudogene_types = {
    "Mt_tRNA_pseudogene", "tRNA_pseudogene", "snoRNA_pseudogene", "snRNA_pseudogene",
    "scRNA_pseudogene", "noncoding_RNA_pseudogene", "misc_RNA_pseudogene", "miRNA_pseudogene",
    "transcribed_processed_pseudogene", "transcribed_unprocessed_pseudogene",
    "transcribed_unitary_pseudogene", "translated_processed_pseudogene",
    "translated_unprocessed_pseudogene", "unitary_pseudogene", "unprocessed_pseudogene",
    "pseudogene", "polymorphic_pseudogene", "processed_pseudogene",
    "IG_C_pseudogene", "IG_J_pseudogene", "IG_V_pseudogene", "IG_pseudogene",
    "TR_V_pseudogene", "TR_J_pseudogene",
}
lncrna_types = {
    "3prime_overlapping_ncRNA", "antisense", "bidirectional_promoter_lncRNA", "lincRNA",
    "macro_lncRNA", "non_coding", "processed_transcript", "sense_intronic",
    "sense_overlapping", "TEC", "lncRNA",
}


def group_gene_type(t):
    if pd.isna(t):
        return None
    t = str(t)
    if t in protein_coding_types:
        return "protein_coding"
    if t in pseudogene_types:
        return "pseudogene"
    if t in lncrna_types:
        return "lncRNA"
    return None


expected_n_bulk_tissues = 54  # histology subtypes excluded; includes 2 cell lines
histology_subtypes = {
    "Colon - Transverse - Muscularis",
    "Stomach - Mixed Cell",
    "Pancreas - Mixed Cell",
    "Small Intestine - Terminal Ileum - Mixed Cell",
    "Stomach - Mucosa",
    "Stomach - Muscularis",
    "Pancreas - Acini",
    "Small Intestine - Terminal Ileum - Lymphoid Aggregate",
    "Colon - Transverse - Mixed Cell",
    "Colon - Transverse - Mucosa",
    "Liver - Mixed Cell",
    "Pancreas - Islets",
    "Liver - Hepatocyte",
    "Liver - Portal Tract",
}
portal_tissue_aliases = {
    "Small_Intestine_Terminal_Ileum_Lymphode_Aggregate": (
        "Small Intestine - Terminal Ileum - Lymphoid Aggregate"
    ),
}


def load_gtex_portal_median_tpm(path: Path) -> pd.DataFrame:
    """Load portal gene_median_tpm GCT; columns = pretty SMTSD names via tissue_name.txt."""
    with gzip.open(path, "rt") as f:
        f.readline()
        f.readline()
        raw = pd.read_csv(f, sep="\t")
    med = raw.set_index("Name").drop(columns=["Description"], errors="ignore")
    tissue_map = pd.read_csv(tissue_names_file, sep="\t")
    stem_to_pretty = dict(zip(tissue_map["tissue_stem"], tissue_map["tissue_name"]))
    renamed = {}
    for c in med.columns:
        pretty = (
            portal_tissue_aliases.get(c)
            or stem_to_pretty.get(c)
            or stem_to_pretty.get(c.replace("-", "_"))
        )
        if pretty is None:
            raise KeyError(f"Cannot map portal median tissue column {c!r} to SMTSD")
        renamed[c] = pretty
    return med.rename(columns=renamed)


biotype_labels = {
    "protein_coding": "protein-coding",
    "lncRNA": "lncRNA",
    "pseudogene": "pseudogene",
}
_paired = sns.color_palette("Paired", 12)
gene_type_palette = {
    "protein_coding": _paired[1],
    "lncRNA": _paired[5],
    "pseudogene": _paired[3],
}


## Paths

In [ ]:
gencode_class_file = Path("path/to/gencode_v39_v47_gene_classification.txt")
tissue_names_file = Path("annotation/refs/tissue_name.txt")
v10_median_tpm = Path("path/to/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_median_tpm.gct.gz")
v11_median_tpm = Path("path/to/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz")
fig_dir = Path("figures")


## Load data


### Gene–biotype map and exon / ambiguity changes (v39 to v47)

In [ ]:
gencode = pd.read_table(gencode_class_file, low_memory=False)
both_ann = (
    gencode.dropna(subset=["gene_id_stable", "gene_id_v39", "gene_id_v47"])
    .drop_duplicates("gene_id_stable")
    .copy()
)
both_ann["biotype_v47"] = both_ann["gene_type_v47"].map(group_gene_type)
both_ann = both_ann[both_ann["biotype_v47"].isin(biotype_order)].copy()

stable_to_biotype = both_ann.set_index("gene_id_stable")["biotype_v47"]
ambig_by_gene = both_ann.set_index("gene_id_stable")[
    ["pct_ambiguous_v39", "pct_ambiguous_v47"]
].astype(float)
exon_change = both_ann.set_index("gene_id_stable")[["pct_exon_turnover"]]

print("both-annotated genes by v47 (v11) biotype:")
print(stable_to_biotype.value_counts().reindex(biotype_order).to_string())


### Portal gene median TPM (v10 and v11; 54 bulk tissues)


In [ ]:
# Assume portal median TPM GCTs are downloaded (paths above); build pairs in memory.
med_v10_all = load_gtex_portal_median_tpm(v10_median_tpm)
med_v11_all = load_gtex_portal_median_tpm(v11_median_tpm)
shared_tissues = sorted(set(med_v10_all.columns) & set(med_v11_all.columns))
tissues = [t for t in shared_tissues if t not in histology_subtypes]
if len(tissues) != expected_n_bulk_tissues:
    raise ValueError(
        f"Expected {expected_n_bulk_tissues} bulk tissues after dropping histology subtypes, "
        f"found {len(tissues)}"
    )
med_v10 = med_v10_all[tissues].copy()
med_v11 = med_v11_all[tissues].copy()
med_v10.index = med_v10.index.astype(str).str.split(".").str[0]
med_v11.index = med_v11.index.astype(str).str.split(".").str[0]
med_v10 = med_v10[~med_v10.index.duplicated(keep="first")]
med_v11 = med_v11[~med_v11.index.duplicated(keep="first")]
genes = sorted(set(med_v10.index) & set(med_v11.index) & set(stable_to_biotype.index))

v10_sub = med_v10.loc[genes, tissues]
v11_sub = med_v11.loc[genes, tissues]
pairs = (
    pd.DataFrame({"tpm_v10": v10_sub.stack(), "tpm_v11": v11_sub.stack()})
    .rename_axis(index=["gene_id_stable", "tissue"])
    .reset_index()
)
pairs["biotype"] = pairs["gene_id_stable"].map(stable_to_biotype)
print(f"gene×tissue pairs: {len(pairs):,}; tissues: {pairs['tissue'].nunique()}")


## Run analysis


### CCC, strong discordance, annotation / ambiguity features


In [ ]:
def lin_ccc(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 2:
        return np.nan
    mx, my = x.mean(), y.mean()
    vx = ((x - mx) ** 2).mean()
    vy = ((y - my) ** 2).mean()
    cov = ((x - mx) * (y - my)).mean()
    denom = vx + vy + (mx - my) ** 2
    return float(2 * cov / denom) if denom > 0 else np.nan


plot_df = pairs.copy()
plot_df["log_tpm_v10"] = np.log10(plot_df["tpm_v10"].clip(lower=0) + 1.0)
plot_df["log_tpm_v11"] = np.log10(plot_df["tpm_v11"].clip(lower=0) + 1.0)
plot_df["log10_fc"] = np.abs(
    np.log10((plot_df["tpm_v11"] + tpm_eps) / (plot_df["tpm_v10"] + tpm_eps))
)
plot_df["strong_discordant"] = plot_df["log10_fc"] >= np.log10(fold_thresh)
plot_df = plot_df.join(ambig_by_gene, on="gene_id_stable")
plot_df = plot_df.join(exon_change, on="gene_id_stable")
plot_df["delta_ambig"] = (
    plot_df["pct_ambiguous_v47"] - plot_df["pct_ambiguous_v39"]
) * 100.0

pooled = {}
pooled_ccc = {}
for biotype in biotype_order:
    sub = plot_df[plot_df["biotype"] == biotype].copy()
    pooled[biotype] = sub
    pooled_ccc[biotype] = lin_ccc(sub["log_tpm_v10"], sub["log_tpm_v11"])
    n_gt = int(sub["strong_discordant"].sum())
    n_genes = sub.loc[sub["strong_discordant"], "gene_id_stable"].nunique()
    n_v10_high = int(
        (sub["strong_discordant"] & (sub["tpm_v10"] > sub["tpm_v11"])).sum()
    )
    n_v11_high = int(
        (sub["strong_discordant"] & (sub["tpm_v11"] > sub["tpm_v10"])).sum()
    )
    print(
        f"{biotype}: {len(sub):,} gene×tissue, "
        f"CCC={pooled_ccc[biotype]:.3f}, "
        f"strong discordant (≥{fold_thresh:g}-fold)={n_gt:,} points / {n_genes:,} genes "
        f"(v10>v11={n_v10_high:,}, v11>v10={n_v11_high:,})"
    )


## Make figure


In [ ]:
# shared axis limit from full data (no clipping)
axis_max = float(
    np.nanpercentile(
        np.concatenate(
            [plot_df["log_tpm_v10"].to_numpy(), plot_df["log_tpm_v11"].to_numpy()]
        ),
        99.9,
    )
)
axis_max = max(axis_max, 1.0)
pad = axis_max * 0.04

# fold-change boundaries on log10(TPM+1) axes (curved, not y=x±c)
tpm_grid = np.logspace(-3, np.log10(10 ** axis_max - 1 + 1e-12), 400)
x_fc = np.log10(tpm_grid + 1.0)
y_hi = np.log10(fold_thresh * (tpm_grid + tpm_eps) - tpm_eps + 1.0)
y_lo = np.log10((tpm_grid + tpm_eps) / fold_thresh - tpm_eps + 1.0)
ok_hi = np.isfinite(y_hi) & (fold_thresh * (tpm_grid + tpm_eps) - tpm_eps >= 0)
ok_lo = np.isfinite(y_lo) & ((tpm_grid + tpm_eps) / fold_thresh - tpm_eps >= 0)

fig, axes = plt.subplots(
    2, 3, figsize=(fig_width, fig_height), dpi=fig_dpi,
    sharex=False, sharey=False, constrained_layout=False,
)

# row a: TPM concordance scatters
for col, biotype in enumerate(biotype_order):
    ax = axes[0, col]
    sub = pooled[biotype]
    edge = gene_type_palette[biotype]
    n_strong = sub.loc[sub["strong_discordant"], "gene_id_stable"].nunique()
    ccc = pooled_ccc[biotype]
    sub_plot = (
        sub.sample(max_scatter, random_state=0) if len(sub) > max_scatter else sub
    )
    x = sub_plot["log_tpm_v10"]
    y = sub_plot["log_tpm_v11"]

    ax.fill_between(
        x_fc[ok_hi], y_hi[ok_hi], axis_max + pad,
        facecolor="0.88", edgecolor="none", alpha=0.9, zorder=0, lw=0,
    )
    ax.fill_between(
        x_fc[ok_lo], -pad, y_lo[ok_lo],
        facecolor="0.88", edgecolor="none", alpha=0.9, zorder=0, lw=0,
    )
    ax.plot(x_fc[ok_hi], y_hi[ok_hi], color="0.45", lw=0.7, ls="--", zorder=1)
    ax.plot(x_fc[ok_lo], y_lo[ok_lo], color="0.45", lw=0.7, ls="--", zorder=1)
    ax.plot([0, axis_max], [0, axis_max], color="0.65", lw=0.5, zorder=1)

    ax.scatter(
        x, y,
        s=3, facecolors="none", edgecolors=edge,
        linewidths=0.2, alpha=0.15, rasterized=True, zorder=2,
    )
    ax.set_xlim(-pad, axis_max + pad)
    ax.set_ylim(-pad, axis_max + pad)
    ax.set_aspect("equal", adjustable="box", anchor="W")
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.tick_params(labelbottom=True, labelleft=True, pad=1.5)
    ax.set_title(
        f"{biotype_labels[biotype]}\n(CCC={ccc:.3f}, {n_strong:,} strongly discordant)",
        fontsize=5.5,
    )
    ax.set_xlabel(r"$v_{10}$ $\log_{10}(\mathrm{TPM}+1)$", labelpad=2)
    ax.set_ylabel(r"$v_{11}$ $\log_{10}(\mathrm{TPM}+1)$", labelpad=2)

axes[0, 0].text(
    -0.28, 1.12, "a",
    transform=axes[0, 0].transAxes,
    va="top", ha="center", fontweight="bold", fontsize=9,
)

# row b: ambiguity vs exon turnover for unique strongly discordant genes
for col, biotype in enumerate(biotype_order):
    ax = axes[1, col]
    edge = gene_type_palette[biotype]
    sub = pooled[biotype]
    strong = (
        sub.loc[
            sub["strong_discordant"],
            ["gene_id_stable", "delta_ambig", "pct_exon_turnover"],
        ]
        .drop_duplicates("gene_id_stable")
        .dropna()
    )
    if len(strong):
        ax.scatter(
            strong["pct_exon_turnover"],
            strong["delta_ambig"],
            s=10,
            marker="o",
            facecolors="none",
            edgecolors=edge,
            linewidths=0.45,
            alpha=0.55,
            rasterized=True,
            zorder=2,
        )
    ax.axhline(0, color="0.55", lw=0.5, ls="--", zorder=1)
    ax.set_xscale("symlog", linthresh=1)
    ax.set_xlim(-0.5, None)
    ax.set_ylim(-105, 105)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.tick_params(labelbottom=True, labelleft=True, pad=1.5)
    ax.set_title(biotype_labels[biotype], fontsize=5.5)
    ax.set_xlabel(r"exon bp unique to one release (% of $v_{47}$)", labelpad=2)
    ax.set_ylabel(r"$v_{47}-v_{39}$ % ambiguous", labelpad=2)

axes[1, 0].text(
    -0.28, 1.12, "b",
    transform=axes[1, 0].transAxes,
    va="top", ha="center", fontweight="bold", fontsize=9,
)

fig.subplots_adjust(wspace=0.45, hspace=0.42, left=0.10, right=0.98, top=0.90, bottom=0.08)
plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "supp_figure_3_v10_v11_tpm_concordance.pdf", bbox_inches="tight")
fig.savefig(fig_dir / "supp_figure_3_v10_v11_tpm_concordance.png", bbox_inches="tight")

## Print stats


In [ ]:
# Caption a: CCC and strongly discordant gene counts by biotype
for biotype in biotype_order:
    sub = pooled[biotype]
    strong = sub[sub["strong_discordant"]]
    print(
        f"{biotype}: CCC={pooled_ccc[biotype]:.3f}, "
        f"strongly discordant genes={strong['gene_id_stable'].nunique():,}"
    )
